# Task 1.3 - Model Comparison Report

Comparing all four models on sentiment analysis (binary: positive/negative) using the 1K and Amazon Polarity dataset from Hugging Face.

## 1. Models Overview

| Model | Type | Parameters | Input Representation | Pre-trained? |
|---|---|---|---|---|
| Simple ANN | Feedforward (2 hidden layers) | 25.7M | TF-IDF vectors (bag-of-words) | No |
| Bi-LSTM | Bidirectional LSTM (2 layers) | ~3M | Word embeddings (learned) | No |
| BERT | Transformer (12 layers) | 110M | WordPiece tokens | Yes (Wikipedia + BookCorpus) |
| DistilBERT | Distilled Transformer (6 layers) | 66M | WordPiece tokens | Yes (distilled from BERT) |

## 2. Results on Large Dataset

| Model | Dataset Size | Epochs | Accuracy | Precision | Recall | F1 | Training Time |
|---|---|---|---|---|---|---|---|
| Simple ANN | 3.6M | 13 | 92.83% | 92.62% | 93.07% | 92.84% | 4 hrs 39 mins |
| Bi-LSTM | 3.6M | 25 | 94.72% | 94.81% | 94.61% | 94.71% | 8 hrs 57 mins |
| **BERT** | **3.6M** | **3** | **96.59%** | **96.54%** | **96.64%** | **96.59%** | **17 hours** |
| DistilBERT | 2.8M | 2 | 96.09% | 96.43% | 95.72% | 96.07% | 2 hrs 13 mins |

## 3. Results on 1K Dataset

| Model | Accuracy | F1 | Training Time |
|---|---|---|---|
| Simple ANN | 78.0% | 78.0% | < 1 min |
| Bi-LSTM | 73.0% | 74.3% | < 1 min |
| **BERT** | **90.0%** | **90.0%** | **27.7s** |
| DistilBERT | 91.0% | 91.1% | 16s |

## 4. Performance Comparison

### Which model performs best and which one prefer one over the other?

BERT achieves the highest accuracy (96.59%) across all models, followed closely by DistilBERT (96.09%). The transformer models outperform both traditional models by a significant margin.

When to prefer each model:
- **BERT**: When accuracy is the top priority and training time/compute is not a constraint.
- **DistilBERT**: When you need near-BERT accuracy but with much faster training (~2 hrs vs ~17 hrs). Best speed/accuracy tradeoff.
- **Bi-LSTM**: When you cannot use pre-trained models or need a lighter-weight solution without the Hugging Face dependency. Still achieves 94.7%.
- **Simple ANN**: When speed and simplicity matter most. Fastest to implement and understand, but lowest accuracy (92.8%).

## 5. Complexity, Accuracy, and Efficiency

### How did the models differ in complexity, accuracy, and efficiency?

| Model | Complexity | Accuracy | Efficiency |
|---|---|---|---|
| Simple ANN | Low — 2 linear layers, TF-IDF input, no sequence understanding | 92.83% | Fast per epoch (24 min), 13 epochs, 4 hrs 39 mins total |
| Bi-LSTM | Medium — bidirectional LSTM, learns word order, trains from scratch | 94.72% | Slow per epoch (21 min), 25 epochs needed, 8 hrs 57 mins total |
| BERT | High — 12-layer transformer, 110M parameters, pre-trained | 96.59% | Slow per epoch (5 hrs 42 mins), but only 3 epochs needed, ~17 hours total |
| DistilBERT | Medium-High — 6-layer transformer, 66M params, distilled from BERT | 96.09% | Fast per epoch (1 hr 7 mins), 2 epochs, 2 hrs 13 mins total |

**Key observation:** BERT has the highest accuracy but also the longest training time. DistilBERT achieves nearly the same accuracy (only 0.50% lower) in about 8x less time. This demonstrates the effectiveness of knowledge distillation — compressing a large model into a smaller one with minimal performance loss.

The Bi-LSTM needed 25 epochs to converge because it trains from scratch with no pre-existing language knowledge. The transformers (BERT/DistilBERT) converge in 2-3 epochs because they already understand language from pre-training — they only need to learn the sentiment classification task.

## 6. Insights on Data Amount, Embeddings, and Architecture

### Data Amount

| Model | 1K Accuracy | Large Dataset Accuracy | Improvement |
|---|---|---|---|
| Simple ANN | 78.0% | 92.8% | +14.8% |
| Bi-LSTM | 73.0% | 94.7% | +21.7% |
| BERT | 90.0% | 96.6% | +6.6% |
| DistilBERT | 91.0% | 96.1% | +5.1% |

- Models trained from scratch (ANN, Bi-LSTM) benefit much more from additional data (+14-21%) compared to pre-trained transformers (+5-7%). This is because BERT and DistilBERT already have language understanding from pre-training, so they perform well even with only 1K samples (90-91%).
- The Bi-LSTM showed the largest improvement (+21.7%) going from 1K to 3.6M, because it needs sequential data to learn word order patterns — impossible with only 800 training samples.
- On 1K data, DistilBERT (91.0%) slightly outperformed BERT (90.0%). With only 100 test samples, this difference is within random variation and shows that the distilled model preserves the teacher's ability even on small data.

### Embeddings

- **Simple ANN** uses TF-IDF (bag-of-words): treats text as a flat vector of word frequencies. Loses word order — "not good" and "good not" are identical. Simple but limited.
- **Bi-LSTM** learns embeddings from scratch during training. Captures word order and sequential patterns. Better than TF-IDF but needs lots of data to learn good representations.
- **BERT/DistilBERT** use WordPiece tokenization with contextualized embeddings. Each word's representation depends on its surrounding context. The word "bank" gets different embeddings in "river bank" vs "bank account". This is the most powerful representation and explains the accuracy gap.

### Architecture

- **Feedforward ANN** processes the entire review as one fixed vector. Cannot understand word order or context. Simplest architecture, lowest accuracy.
- **Bi-LSTM** reads the review sequentially in both directions (forward and backward). Understands word order — "not good" is different from "good". But information from early words can fade over long reviews (vanishing gradient problem).
- **Transformers (BERT/DistilBERT)** use self-attention to relate every word to every other word simultaneously. No information loss over distance. Combined with pre-training on massive text corpora, this gives them the best accuracy with the fewest training epochs.

## 7. Convergence and Overfitting Analysis

### Convergence
- All four models converged on the large dataset. The transformers converged fastest (2-3 epochs) due to pre-training, while ANN needed 13 epochs and Bi-LSTM needed 25 epochs.
- On the 1K dataset, only the transformers converged meaningfully. ANN and Bi-LSTM did not have enough data to learn robust patterns.

### Overfitting
- **1K dataset:** All models showed overfitting. BERT's validation loss decreased throughout all 5 epochs (0.399 → 0.188 → 0.188 → 0.175 → 0.170), but validation accuracy at epoch 5 (95%) was much higher than test accuracy (90%), indicating the model overfit to the tiny validation set (100 samples). ANN achieved 99% train accuracy but only 78% test accuracy — a clear sign of memorization.
- **Large dataset:** BERT showed mild overfitting at epoch 3 (validation loss increased from 0.111 to 0.123, an 11% increase). ANN overfit more severely (train accuracy 97% vs test accuracy 92.8%). Bi-LSTM was the most stable with gradual convergence over 25 epochs.
- All models used early stopping and best model selection (by lowest validation loss or highest validation accuracy) to mitigate overfitting.

### Best Model Selection
- BERT and DistilBERT used `load_best_model_at_end` with lowest validation loss as the selection criterion.
- ANN and Bi-LSTM saved the best checkpoint based on validation loss and loaded it for final evaluation.

## 8. Summary

| Rank | Model | Large Dataset F1 | 1K F1 | Training Time | Best For |
|---|---|---|---|---|---|
| 1 | BERT | 96.59% | 90.0% | 17 hours | Maximum accuracy |
| 2 | DistilBERT | 96.07% | 91.1% | 2 hrs 13 mins | Best speed/accuracy tradeoff |
| 3 | Bi-LSTM | 94.71% | 74.3% | 8 hrs 57 mins | When pre-trained models unavailable |
| 4 | Simple ANN | 92.84% | 78.0% | 4 hrs 39 mins | Simplicity and speed |

Pre-trained transformer models (BERT, DistilBERT) consistently outperform traditional models across all dataset sizes. The advantage is most pronounced on small data (1K) where pre-training compensates for limited training examples. DistilBERT offers the best practical tradeoff — 96% accuracy in just over 2 hours of training, compared to BERT's 17-hour run.